# Piloto EFT-SP — VMamba Small

200 atualizações, full fine-tuning, AdamW lr=1e-5, sem weight decay, batch 1, float32, CE + alinhamento. Último estágio com normalização pré-treinada, entrada 256 e grade ALP 32×32.

Este piloto usa **200 imagens distintas** da divisão de treino, uma região escolhida por imagem. A validação usa uma região fixa por imagem elegível reservada, antes e depois. Aumentações limitadas ao flip horizontal da mesma imagem/região, como no diagnóstico. Não é treino completo nem reprodução exata da receita do artigo.

Na validação, `eval()` desativa alinhamento e drop path. `isval=False` mantém o kernel ALP 4×4. Registramos CE e Dice de foreground de superpixels: não são métricas de pólipos. Não selecionamos checkpoints pelas imagens de teste.

Execute uma variante por vez. Comece pelo Tiny; envie os resultados antes de iniciar Small/Base.

**O piloto mantém os pesos adaptados em `checkpoint_latest.pt`.** Os checkpoints pré-treinados de origem nunca são alterados. A gravação ocorre antes do treino e a cada 25 atualizações; não há uma cópia por passo. A retomada refaz somente passos posteriores à última gravação.


In [ ]:
from pathlib import Path
import sys

ESPERADO = "/workspace/miniforge3/envs/protosam/bin/python"
print("Python do kernel:", sys.executable)
assert Path(sys.executable).absolute() == Path(ESPERADO), "Selecione Kernel → Change Kernel → Python (protosam)."
REPO = Path("/workspace/ProtoSAM-repro")
PASTA = REPO / "exp/eft_sp_piloto_v1"
assert (PASTA / "eft_dados.py").is_file(), PASTA
for caminho in (REPO, PASTA):
    if str(caminho) not in sys.path:
        sys.path.insert(0, str(caminho))


In [ ]:
from pathlib import Path
from eft_treino import executar_piloto

PASTA_DADOS = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "eft_sp_piloto_dados_20261011T152315570066Z"
)
assert (PASTA_DADOS / "planos.json").is_file()

VARIANTE = "small"
RETOMAR = None

In [ ]:
PASTA_EXECUCAO = executar_piloto(
    dados=PASTA_DADOS, variante=VARIANTE, retomar=RETOMAR, repo=REPO,
)
print("RESULTADOS:", PASTA_EXECUCAO)


Envie o resumo final. Se houver erro, envie o traceback completo e o caminho EXECUÇÃO. Não mude filtros, loss ou taxa de aprendizado para contornar uma falha sem registrá-la. Em uma retomada, logs são separados por tentativa e indicam o passo inicial. Mesmo com RNG salvo, determinismo bit a bit dos kernels CUDA não foi comprovado.